# 01 · 真实数据与配置读取(★★★★★)

学会:`YParams` 读 config、`ERA5Datapipe` 出真实 batch、静态场加载、**99→69 按名字对齐**、连续 6h 配对。

> 关键结论(已实测):`data` 是长度 5 的 list —— `data[0]`=输入 [1,69,721,1440](**已归一**)、`data[1]`=标签(下一6h)、`data[4]`=[[输入时刻],[目标时刻]] 文件名。

In [ ]:
# ============ 公共设置(每个 notebook 先跑这一格)============
import os, sys, json, glob, math, time, numpy as np, torch, torch.nn.functional as F
import warnings; warnings.filterwarnings("ignore")

BASE = "/public/home/xdzs2026_c296"          # ★你的主目录,若不同改这里
BASELINE = f"{BASE}/xiandao2026-AI4S/pangu_weather"   # 官方 baseline(含 maxvit3d_student.py, conf, data)
CKPT = f"{BASELINE}/data/checkpoints/model_bak.pth"   # 教师权重
TRAIN_DATA = f"{BASE}/era5_real"              # 训练数据(13年)
VAL_DATA   = f"{BASE}/era5_testc"             # 验证/测试数据(2000年)
WORK = f"{BASE}/_learn_work"                  # 本教程的工作目录(存中间文件)
os.makedirs(WORK, exist_ok=True)
sys.path.insert(0, BASELINE)                  # 为了 import maxvit3d_student
print("torch", torch.__version__, "| DCU 可用:", torch.cuda.is_available())


In [ ]:
# 造一个指向验证数据的 conf/config.yaml(官方 config 的 data_dir 指向 /work2 拿不到,改指 VAL_DATA)
os.makedirs(f"{WORK}/conf", exist_ok=True)
src = open(f"{BASELINE}/conf/config.yaml", encoding="utf-8").read()
src = src.replace("/work2/share/sugonhpcapp01/ERA5/old-data", VAL_DATA)
open(f"{WORK}/conf/config.yaml", "w", encoding="utf-8").write(src)
print("已写", f"{WORK}/conf/config.yaml", "-> data_dir =", VAL_DATA)


## 1) YParams 读配置

In [ ]:
from onescience.utils.YParams import YParams
cfg = YParams(f"{WORK}/conf/config.yaml", "datapipe")
print("data_dir:", cfg.dataset.data_dir)
print("通道数:", len(cfg.dataset.channels), "| 前4:", cfg.dataset.channels[:4])
print("static_dir:", cfg.dataset.static_dir, "| test年份:", cfg.dataset.test_ratio, "| batch:", cfg.dataloader.batch_size)
channels = list(cfg.dataset.channels)

## 2) ERA5Datapipe 出一个真实 batch(看结构/形状/文件名)

In [ ]:
from onescience.datapipes.climate import ERA5Datapipe
dp = ERA5Datapipe(params=cfg, distributed=False)
dl = dp.test_dataloader()
for data in dl:
    print("data 是 list,长度", len(data))
    print("  data[0] 输入 :", tuple(data[0].shape), data[0].dtype, " (已归一的 69 通道)")
    print("  data[1] 标签 :", tuple(data[1].shape), " (下一个 6h 的场)")
    print("  data[4] 文件名:", data[4], " -> 目标时刻 =", data[4][-1][0])
    invar, outvar, filename = data[0], data[1], data[4][-1][0]
    break

## 3) 静态场加载 + 地形归一化(3 通道:land_mask/soil_type/topography)

In [ ]:
sd = cfg.dataset.static_dir
land = torch.from_numpy(np.load(f"{sd}/land_mask.npy").astype(np.float32))
soil = torch.from_numpy(np.load(f"{sd}/soil_type.npy").astype(np.float32))
topo = torch.from_numpy(np.load(f"{sd}/topography.npy").astype(np.float32))
topo = (topo - topo.mean()) / (topo.std(unbiased=False) + 1e-6)   # 地形归一化
surface_mask = torch.stack([land, soil, topo], 0).unsqueeze(0)     # [1,3,721,1440]
print("静态场:", tuple(surface_mask.shape))

## 4) ★99→69 按名字对齐(考点!不能取前 69)+ mean/std 用同索引

h5 里是 **99 通道**;config 要的 69 个通道要**按变量名**去 metadata 查下标(不是 `range(69)`)。均值/标准差用**同一组下标**。

In [ ]:
meta = json.load(open(f"{cfg.dataset.data_dir}/metadata.json"))["variables"]
sel = [meta.index(v) for v in channels]           # 69 个在 99 通道里的真实下标
print("metadata 总变量:", len(meta), "| 选出69的前5下标:", sel[:5], "(注意不是 0,1,2,3,4)")
mu = np.load(f"{cfg.dataset.data_dir}/stats/global_means.npy")   # (1,99,1,1)
sdv = np.load(f"{cfg.dataset.data_dir}/stats/global_stds.npy")
mu69, sd69 = mu[:, sel], sdv[:, sel]              # 同索引选 69
print("global_means:", mu.shape, "-> 选69:", mu69.shape)

## 5) 手动构造模型输入(72 通道 = 4预测面 + 3静态 + 65高空)+ 连续6h配对

推理/自建训练时常这样手动读:读 h5 → 按名字选 69 → 归一化 → 插入 3 静态场 → 得 72 通道。

In [ ]:
def read69(h5path):
    import h5py
    with h5py.File(h5path, "r") as h: fld = h["fields"][:].astype(np.float32)
    return fld[sel] if fld.shape[0] != 69 else fld     # 99->69 按名字

files = sorted(glob.glob(f"{VAL_DATA}/data/2000/*.h5"))
from datetime import datetime
def ts(f): return datetime.strptime(os.path.basename(f)[:10], "%Y%m%d%H")
pairs = [(files[i], files[i+1]) for i in range(len(files)-1)
         if abs((ts(files[i+1]) - ts(files[i])).total_seconds() - 21600) < 60]   # 真6h连续对
print("连续6h对:", len(pairs))
raw = read69(pairs[0][0])                      # [69,721,1440] 物理量
norm = (raw - mu69.reshape(69,1,1)) / sd69.reshape(69,1,1)      # 归一化
x72 = np.concatenate([norm[:4], surface_mask[0].numpy(), norm[4:]], 0)   # 4+3+65=72
print("手动输入 x72:", x72.shape, " (喂给学生/教师的就是这个)")
assert x72.shape[0] == 72

### ✅ 本节要点
- data[0] 是**已归一的 69 通道**,datapipe 已帮你选好通道+归一。
- 手动路径:读 h5(99)→ **按名字选 69** → 归一化 → 插 3 静态 → 72。
- 均值/标准差与通道选择**用同一组下标**。这是评测不出错的关键(见 06)。